In [0]:
%fs
ls dbfs:/Volumes/workspace/training_samples/test/GR/


In [0]:
%fs ls  dbfs:/Volumes/workspace/training/volume01

In [0]:
%sh ls -dR

In [0]:
%sh ls -d -R "$PWD/"data/*

## Bronze

In [0]:
spark.sql("create database if not exists globalretail_bronze")

In [0]:
%sql
show databases;

In [0]:
%sql
use database globalretail_bronze;

### load data from files
* data files needs to be copied into dbfs volume dbfs:/Volumes/workspace/training_samples/test/GR/ 




In [0]:
%fs ls dbfs:/Volumes/workspace/training_samples/test/GR/ 



In [0]:
%sh python --version

In [0]:




def create_table( filePath,filename, fformat, tablename):
    match fformat:
        case "csv":     
            spark.read.format(fformat).\
                option("header", "true").\
                option("inferSchema", "true").\
                load(filePath+filename).\
                write.\
                mode("overwrite").\
                saveAsTable(tablename)
        case "parquet":     
            spark.read.format(fformat).\
                option("header", "true").\
                load(filePath+filename).\
                write.\
                mode("overwrite").\
                saveAsTable(tablename)        
    return True


d= [
    {'filename'  :"customer.csv",
    'filePath'  :f"dbfs:/Volumes/workspace/training_samples/test/GR/",
    'fformat'   : "csv",
    'tablename' : "globalretail_bronze.customer"},

    {'filename'  :"sales_data.csv",
    'filePath'  :f"dbfs:/Volumes/workspace/training_samples/test/GR/",
    'fformat'   : "csv",
    'tablename' : "globalretail_bronze.sales_data"},  

    {'filename'  :"sales_data.csv",
    'filePath'  :f"dbfs:/Volumes/workspace/training_samples/test/GR/",
    'fformat'   : "csv",
    'tablename' : "globalretail_bronze.sales_data"},  

    {'filename'  :"transactions.snappy.parquet",
    'filePath'  :f"dbfs:/Volumes/workspace/training_samples/test/GR/",
    'fformat'   : "parquet",
    'tablename' : "globalretail_bronze.transactions"},
    
    ]


dict (list (map ( lambda x : ( x.get("tablename") ,create_table(**x) ) , d )))

In [0]:
dict(l)

In [0]:
%scala
def createTable(filePath: String, filename: String, fformat: String, tablename: String): Boolean = {
  fformat match {
    case "csv" =>
      spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(filePath + filename)
        .write
        .mode("overwrite")
        .saveAsTable(tablename)

    case "parquet" =>
      spark.read.format("parquet")
        .option("header", "true") // harmless for parquet
        .load(filePath + filename)
        .write
        .mode("overwrite")
        .saveAsTable(tablename)

    case _ =>
      throw new IllegalArgumentException(s"Unsupported format: $fformat")
  }
  true
}

// List of tables to create
val d = List(
  Map("filename" -> "customer.csv",
      "filePath" -> "dbfs:/Volumes/workspace/training_samples/test/GR/",
      "fformat"  -> "csv",
      "tablename"-> "globalretail_bronze.customer"),

  Map("filename" -> "sales_data.csv",
      "filePath" -> "dbfs:/Volumes/workspace/training_samples/test/GR/",
      "fformat"  -> "csv",
      "tablename"-> "globalretail_bronze.sales_data"),

  Map("filename" -> "transactions.snappy.parquet",
      "filePath" -> "dbfs:/Volumes/workspace/training_samples/test/GR/",
      "fformat"  -> "parquet",
      "tablename"-> "globalretail_bronze.transactions")
)

// Apply the function to each map
val results: Map[String, Boolean] = d.map { x =>
  val filename  = x("filename")
  val filePath  = x("filePath")
  val fformat   = x("fformat")
  val tablename = x("tablename")
  (tablename, createTable(filePath, filename, fformat, tablename))
}.toMap

println(results)


In [0]:
%sql 
drop table globalretail_bronze.customer_